In [8]:
# ruff: noqa: T201, T203
from __future__ import annotations

import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import logging

import pandas as pd
from dotenv import load_dotenv
from tqdm import tqdm

import src.prompts.system as system_prompts
from config.config import RUN_DIR
from config.runs_vars import SUBSET_TO_DATASET_MAP
from src.constants import PAIRS_SEPARATOR
from src.evaluate import (
    analyze_results,
    get_predictions_with_gt,
    plot_usage_histograms,
    read_run_metrics_df,
    store_run_metrics_df,
)
from src.formatting import (
    format_combined_metrics_path,
    format_oracle_pairs_filepath,
    format_oracle_pairs_precomputed_dir,
    format_predictions_run_path,
    format_run_path,
    format_storing_pathes_from_run_path,
    format_subsets_ontologies_paths,
)
from src.LLM_servers.openai import OpenAIServer
from src.onto_access import OntologyAccess
from src.onto_object import OntologyEntryAttr
from src.processing import parallel_samples_process, save_oracle_pairs_with_prompts, try_load_precomputed_oracle_pairs
from src.prompts.prompts import (
    prompt_direct_entity,
    prompt_direct_entity_ontological,
    prompt_direct_entity_with_synonyms,
    prompt_sequential_hierarchy,
    prompt_sequential_hierarchy_ontological,
    prompt_sequential_hierarchy_with_synonyms,
)
from src.utils import read_oracle_pairs, save_run_results

pd.set_option("display.max_rows", None)
logging.getLogger().setLevel(logging.WARNING)
load_dotenv()

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [9]:
PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        prompt_direct_entity,
        prompt_direct_entity_ontological,
        prompt_sequential_hierarchy,
        prompt_sequential_hierarchy_ontological,
        prompt_direct_entity_with_synonyms,
        prompt_sequential_hierarchy_with_synonyms,
    ]
}
# ruff: noqa: ERA001
# SYSPROMPTS_MAP = {
#     "base": system_prompts.BASELINE_INITIALIZATION_MESSAGE,
#     "natural_language": system_prompts.INTUITIVE_NATURAL_LANGUAGE_JUDGEMENT_MESSAGE,
#     "ontology_aware": system_prompts.ONTOLOGY_AWARE_REASONING_MESSAGE,
#     "synonym_aware": system_prompts.SYNONYM_AWARE_MESSAGE,
#     "none": None,
# }

In [3]:
# DATASET, SUBSET = "anatomy", "human-mouse"
DATASET, SUBSET = "largebio", "fma-nci"

src_onto_path, tgt_onto_path = format_subsets_ontologies_paths(DATASET, SUBSET)
src_onto_path, tgt_onto_path

(PosixPath('data/largebio/fma-nci/oaei_FMA_whole_ontology.owl'),
 PosixPath('data/largebio/fma-nci/oaei_NCI_whole_ontology.owl'))

### Pre-Validation

In [ ]:
onto_src = OntologyAccess(src_onto_path, annotate_on_init=True)
onto_tgt = OntologyAccess(tgt_onto_path, annotate_on_init=True)

In [ ]:
src_entity, tgt_entity = read_oracle_pairs(format_oracle_pairs_filepath(DATASET, SUBSET))[0]
src_entity, tgt_entity = OntologyEntryAttr(src_entity, onto_src), OntologyEntryAttr(tgt_entity, onto_tgt)

print(src_entity)
print(tgt_entity)

for name, prompt_function in PROMPT_FUNCTIONS_MAP.items():
    print(f"Prompt function: {name}")
    print(prompt_function(src_entity, tgt_entity))
    print("=" * 50)

In [ ]:
for name, prompt_function in PROMPT_FUNCTIONS_MAP.items():
    print(f"Prompt function: {name}")
    oracle_candidate_pairs = try_load_precomputed_oracle_pairs(DATASET, SUBSET, prompt_function)
    print(*oracle_candidate_pairs[0], sep="\n")


### Pre-Processing

In [86]:
ALL_DATASET_NAMES = {
    # "anatomy": ["human-mouse"]j,
    # "bioml-2024": ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", "ncit-doid", "omim-ordo"],
    "largebio": ["fma-snomed"] #, "snomed-nci"], #"fma-nci"
}

PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        # prompt_direct_entity,
        # prompt_direct_entity_ontological,
        # prompt_sequential_hierarchy,
        # prompt_sequential_hierarchy_ontological,
        prompt_direct_entity_with_synonyms,
        # prompt_sequential_hierarchy_with_synonyms,
    ]
}
prompt_functions = list(PROMPT_FUNCTIONS_MAP.values())

for dataset_name, set_names in ALL_DATASET_NAMES.items():
    for set_name in set_names:
        try:
            # ontologies = []
            # for onto_path in tqdm(
            #     format_subsets_ontologies_paths(dataset_name, set_name),
            #     desc=f"Loading ontologies: {dataset_name} - {set_name}",
            # ):
            #     ontologies.append(OntologyAccess(onto_path, annotate_on_init=True))

            pairs_path = format_oracle_pairs_filepath(dataset_name, set_name)
            oracle_pairs_dir = format_oracle_pairs_precomputed_dir(dataset_name, set_name)

            pairs_with_prompts_df = save_oracle_pairs_with_prompts(
                pairs_path, *ontologies, prompt_functions, oracle_pairs_dir, PAIRS_SEPARATOR, max_workers=2
            )
        except Exception as e:  # noqa: BLE001
            print(f"Error processing {dataset_name} - {set_name}: {e}")
            continue

Adding prompts:   0%|          | 0/5284 [00:00<?, ?it/s]

Class URI: http://bioontology.org/projects/ontologies/fma/fmaOwlDlComponent_2_0#Right_inferior_parathyroid_gland


Adding prompts:   0%|          | 1/5284 [00:00<35:47,  2.46it/s]

Class URI: http://www.ihtsdo.org/snomed#Structure_of_right_inferior_parathyroid_gland
Class URI: None
Extracting parent name
Class URI: None
Extracting parent name
Parent names: Homogeneous organ, Thing
Entity names: Right lower parathyroid gland, Right inferior parathyroid gland, Structure_of_right_inferior_parathyroid_gland
Class URI: http://bioontology.org/projects/ontologies/fma/fmaOwlDlComponent_2_0#Right_midclavicular_line


Adding prompts:   0%|          | 2/5284 [00:00<36:46,  2.39it/s]

...

Adding prompts:   1%|          | 34/5284 [00:13<37:30,  2.33it/s]

Class URI: http://www.ihtsdo.org/snomed#T7_T8_ligamentum_flavum
Class URI: None
Extracting parent name
Class URI: None
Extracting parent name
Parent names: Ligament organ, Thoracic_ligamentum_flavum
Entity names: Ligamentum flavum of T7-T8 vertebrae, T7_T8_ligamentum_flavum
Class URI: http://bioontology.org/projects/ontologies/fma/fmaOwlDlComponent_2_0#Anulus_fibrosus_of_intervertebral_disk_of_third_thoracic_vertebra


Adding prompts:   1%|          | 35/5284 [00:14<35:39,  2.45it/s]

Class URI: http://www.ihtsdo.org/snomed#Structure_of_anulus_fibrosus_of_intervertebral_disc_of_third_thoracic_vertebra
Class URI: None
Extracting parent name
Class URI: None
Extracting parent name
Parent names: Biological structure, Anatomical structure, Intervertebral_disc_structure_of_third_thoracic_vertebra
Entity names: Anulus fibrosus of intervertebral disk of third thoracic vertebra, Structure_of_anulus_fibrosus_of_intervertebral_disc_of_third_thoracic_vertebra


KeyboardInterrupt: 

In [10]:
prompt_functions = list(PROMPT_FUNCTIONS_MAP.values())
dataset_name, set_name = "largebio", "fma-snomed"

ontologies = []

for onto_path in tqdm(
    format_subsets_ontologies_paths(dataset_name, set_name), desc=f"Loading ontologies: {dataset_name} - {set_name}"
):
    ontologies.append(OntologyAccess(onto_path, annotate_on_init=True))

pairs_path = format_oracle_pairs_filepath(dataset_name, set_name)
oracle_pairs_dir = format_oracle_pairs_precomputed_dir(dataset_name, set_name)

Loading ontologies: largebio - fma-snomed:   0%|          | 0/2 [00:00<?, ?it/s]

Loading ontologies: largebio - fma-snomed:  50%|█████     | 1/2 [04:45<04:45, 285.43s/it]* Owlready2 * Creating new ontology oaei_SNOMED_whole_ontology <data/largebio/fma-snomed/oaei_SNOMED_whole_ontology.owl#>.
* Owlready2 * ADD TRIPLE data/largebio/fma-snomed/oaei_SNOMED_whole_ontology.owl http://www.w3.org/1999/02/22-rdf-syntax-ns#type http://www.w3.org/2002/07/owl#Ontology
* Owlready2 *     ...loading ontology oaei_SNOMED_whole_ontology from data/largebio/fma-snomed/oaei_SNOMED_whole_ontology.owl...
* Owlready2 *     ...56 properties found: Access, After, Associated_with, Associated_finding, Associated_morphology, Associated_procedure, Causative_agent, Clinical_course, Component, Direct_device, Procedure_device, Direct_morphology, Procedure_morphology, Direct_substance, Has_active_ingredient, Due_to, Finding_context, Finding_informer, Finding_method, Finding_site, Has_definitional_manifestation, Has_dose_form, Has_focus, Has_intent, Has_interpretation, Has_specimen, Indirect_device

In [31]:
pairs[0][1]

'http://www.ihtsdo.org/snomed#Cardiac_chamber_structure'

In [72]:
onto_src, onto_tgt = ontologies
pairs = try_load_precomputed_oracle_pairs(DATASET, "fma-snomed", prompt_function=None)
src_entity = OntologyEntryAttr(pairs[0][0], onto_src)

Ontology loaded: get_ontology("http://bioontology.org/projects/ontologies/fma/fmaOwlDlComponent_2_0#")
Class URI: http://bioontology.org/projects/ontologies/fma/fmaOwlDlComponent_2_0#Cardiac_chamber


In [84]:
tgt_entity = OntologyEntryAttr(pairs[12][1], onto_tgt)

Class URI: http://www.ihtsdo.org/snomed#Structure_of_hypoglossal_canal


In [56]:

# print(src_entity)
# print(tgt_entity)

PROMPT_FUNCTIONS_MAP = {
    prompt_function.__name__.replace("prompt_", ""): prompt_function
    for prompt_function in [
        # prompt_direct_entity,
        # prompt_direct_entity_ontological,
        # prompt_sequential_hierarchy,
        # prompt_sequential_hierarchy_ontological,
        prompt_direct_entity_with_synonyms,
        # prompt_sequential_hierarchy_with_synonyms,
    ]
}


for name, prompt_function in PROMPT_FUNCTIONS_MAP.items():
    print(f"Prompt function: {name}")
    try:
        print(prompt_function(src_entity, tgt_entity))
    except Exception as e:  # noqa: BLE001
        print(f"Error processing {dataset_name=} - {set_name=} - {prompt_function=}: {e}")
        raise e
        continue
    print("=" * 50)

Prompt function: direct_entity_with_synonyms
Extracting parent name
Extracting parent name
Parent names: Biological structure, Anatomical structure, Structure_of_lung_and_or_mediastinum
Entity names: Cardiac chamber, Chamber of heart, Cardiac_chamber_structure
We have two entities from different biomedical ontologies.
The first one is "Cardiac chamber, Chamber of heart", also known as "Cardiac_chamber", which falls under the category "Biological structure, Anatomical structure".
The second one is "Cardiac_chamber_structure", also known as "Cardiac_chamber_structure", which falls under the category "Structure_of_lung_and_or_mediastinum".

Do they mean the same thing? Respond with "True" or "False".


In [23]:
try:
    pairs_with_prompts_df = save_oracle_pairs_with_prompts(
        pairs_path, *ontologies, prompt_functions, oracle_pairs_dir, PAIRS_SEPARATOR, max_workers=2
    )
except Exception as e:  # noqa: BLE001
    raise e
    print(f"Error processing {dataset_name} - {set_name}: {e!s}")

Adding prompts:   0%|          | 0/5284 [00:00<?, ?it/s]

* Owlready2 * Warning: ignoring cyclic subclass of/subproperty of, involving:
  http://www.ihtsdo.org/snomed#Cinchona_alkaloid
  http://www.ihtsdo.org/snomed#Cinchona_antimalarial

* Owlready2 * Warning: ignoring cyclic subclass of/subproperty of, involving:
  http://www.ihtsdo.org/snomed#alpha_Thalassemia

* Owlready2 * Warning: ignoring cyclic subclass of/subproperty of, involving:
  http://www.ihtsdo.org/snomed#beta_Thalassemia

* Owlready2 * Warning: ignoring cyclic subclass of/subproperty of, involving:
  http://www.ihtsdo.org/snomed#Cinchona_alkaloid
  http://www.ihtsdo.org/snomed#Cinchona_antimalarial

* Owlready2 * Warning: ignoring cyclic subclass of/subproperty of, involving:
  http://www.ihtsdo.org/snomed#alpha_Thalassemia

* Owlready2 * Warning: ignoring cyclic subclass of/subproperty of, involving:
  http://www.ihtsdo.org/snomed#beta_Thalassemia

Adding prompts:   0%|          | 0/5284 [00:38<?, ?it/s]


TypeError: a __bases__ item causes an inheritance cycle

### Run experiments


In [4]:
MAX_WORKERS = 10
MODELS = ["gemini-2.0-flash"]  # "gpt-4o-mini"

DATASETS_MAP = {
    # "anatomy": ["human-mouse"],
    # "bioml-2024": ["omim-ordo"], # ["snomed-fma.body", "snomed-ncit.neoplas", "snomed-ncit.pharm", # "ncit-doid", "omim-ordo"],
    "largebio": ["fma-nci"], # "fma-snomed", "snomed-nci"]
}
EXPERIMENT_TYPE = "prompts"
EXP_NAMES_MAP = [""]

onto_src, onto_tgt = None, None

In [7]:
for exp_name in EXP_NAMES_MAP:
    for DATASET in DATASETS_MAP:
        for SUBSET in DATASETS_MAP[DATASET]:
            # Load the ontologies here, if there are no precomputed prompts
            for MODEL in MODELS:
                llm_oracle = (
                    OpenAIServer(
                        api_key=os.environ["GEMINI_API_KEY"],
                        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
                    )
                    if MODEL.startswith("gemini")
                    else OpenAIServer(api_key=os.environ["OPENAI_API_KEY"])
                )
                llm_oracle.add_system_context(system_prompts.INTUITIVE_NATURAL_LANGUAGE_JUDGEMENT_MESSAGE)

                # run_path = format_predictions_run_path(DATASET, SUBSET, MODEL, EXPERIMENT_TYPE, exp_spec=exp_name)
                run_path = format_run_path()
                print(f"{run_path=} | {DATASET=} | {SUBSET=} | {MODEL=} | {exp_name=}")

                for prompt_name, prompt_function in PROMPT_FUNCTIONS_MAP.items():
                    oracle_candidate_pairs = try_load_precomputed_oracle_pairs(DATASET, SUBSET, prompt_function)

                    results, tokens_usage, confidences = parallel_samples_process(
                        oracle_candidate_pairs, llm_oracle, onto_src, onto_tgt, MODEL, MAX_WORKERS, prompt_function
                    )
                    prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
                        run_path, SUBSET, MODEL, prompt_name, suffix=""
                    )
                    save_run_results(results, prediction_path, columns=["Source", "Target", "Prediction", "Confidence"])
                    plot_usage_histograms(
                        tokens_usage, confidences, do_plot=False, do_print=False, suptitle=prompt_name
                    )
                    try:
                        predictions = get_predictions_with_gt(run_path, DATASET, SUBSET, MODEL, prompt_name, suffix="")
                        analyze_results(
                            predictions,
                            print_results=False,
                            plot_confusion_matrix=False,
                            subtitle=f"{SUBSET}: {MODEL} {prompt_name} | ",
                            cm_save_path=diagram_path,
                            stats_path=stats_path,
                        )
                    except Exception as e:  # noqa: BLE001
                        print(f"Error: {e}")

                store_run_metrics_df(PROMPT_FUNCTIONS_MAP, run_path, DATASET, SUBSET, MODEL)

run_path=PosixPath('runs/2025-05-08_15-29-28') | DATASET='largebio' | SUBSET='fma-nci' | MODEL='gemini-2.0-flash' | exp_name=''


Processing Lines prompt_direct_entity: 100%|██████████| 1258/1258 [00:57<00:00, 21.87it/s]
Processing Lines prompt_direct_entity_ontological: 100%|██████████| 1258/1258 [00:52<00:00, 23.82it/s]
Processing Lines prompt_sequential_hierarchy: 100%|██████████| 1258/1258 [00:54<00:00, 23.20it/s]
Processing Lines prompt_sequential_hierarchy_ontological: 100%|██████████| 1258/1258 [00:52<00:00, 23.97it/s]
Processing Lines prompt_direct_entity_with_synonyms: 100%|██████████| 1258/1258 [00:51<00:00, 24.20it/s]
Processing Lines prompt_sequential_hierarchy_with_synonyms: 100%|██████████| 1258/1258 [00:52<00:00, 23.89it/s]


## Analysis


In [ ]:
runs_array = []

for directory in sorted(RUN_DIR.iterdir(), key=lambda x: x.name, reverse=True):
    if not directory.is_dir():
        continue
    for file in directory.iterdir():
        if '_' not in file.name:
            continue
        subset_name, model_name = file.name.split('_')[:2]
        dataset_name = SUBSET_TO_DATASET_MAP.get(subset_name)
    runs_array.append((directory.name, model_name, subset_name, dataset_name))

len(runs_array), runs_array

In [ ]:
runs = [
    ('2025-05-01_11-47-16', 'gemini-2.0-flash', 'snomed-nci', 'largebio'),
    ('2025-05-01_11-37-30', 'gemini-2.0-flash-lite', 'snomed-nci', 'largebio'),
    ('2025-05-01_11-24-31', 'gemini-2.0-flash', 'fma-snomed', 'largebio'),
    ('2025-05-01_11-19-48', 'gemini-2.0-flash-lite', 'fma-snomed', 'largebio'),
    ('2025-05-01_11-15-05', 'gemini-2.0-flash', 'snomed-ncit.pharm', 'bioml-2024'),
    ('2025-05-01_11-12-35', 'gemini-2.0-flash-lite', 'snomed-ncit.pharm', 'bioml-2024'),
    ('2025-05-01_11-07-00', 'gemini-2.0-flash', 'snomed-ncit.neoplas', 'bioml-2024'),
    ('2025-05-01_11-06-39', 'gemini-2.0-flash-lite', 'snomed-ncit.neoplas', 'bioml-2024'),
    ('2025-05-01_10-56-46', 'gemini-2.0-flash', 'snomed-fma.body', 'bioml-2024'),
    ('2025-05-01_10-50-33', 'gemini-2.0-flash-lite', 'snomed-fma.body', 'bioml-2024'),
    ('2025-05-01_10-13-02', 'gemini-2.0-flash-lite', 'fma-snomed', 'largebio'),
    ('2025-04-10_22-11-36', 'gemini-2.0-flash', 'ncit-doid', 'bioml-2024'),
    ('2025-04-10_00-33-19', 'gemini-2.0-flash-lite', 'ncit-doid', 'bioml-2024'),
    ('2025-04-06_22-05-02', 'gemini-2.0-flash-lite', 'omim-ordo', 'bioml-2024'),
    ('2025-04-06_19-27-41', 'gemini-2.0-flash', 'omim-ordo', 'bioml-2024'),
    ('2025-04-06_14-51-00', 'gemini-2.0-flash-lite', 'fma-nci', 'largebio'),
    ('2025-04-06_14-18-46', 'gemini-2.0-flash', 'fma-nci', 'largebio'),
    ('2025-04-04_19-41-28', 'gemini-2.0-flash', 'human-mouse', 'anatomy'),
    ('2025-04-04_19-23-47', 'gemini-2.0-flash-lite', 'human-mouse', 'anatomy'),
    ('2025-04-04_00-43-16', 'gemini-2.0-flash', 'fma-snomed', 'largebio'),
    ('2025-03-28_10-15-04', 'gpt-4o-mini', 'fma-nci', 'largebio'),
    ('2025-03-19_23-32-20', 'gpt-4o-mini', 'omim-ordo', 'bioml-2024'),
    ('2025-03-19_22-17-52', 'gpt-4o-mini', 'ncit-doid', 'bioml-2024'),
    ('2025-03-19_19-48-41', 'gpt-4o-mini', 'human-mouse', 'anatomy'),
    ('2025-03-14_00-40-55', 'gpt-4o-mini', 'ncit-doid', 'bioml-2024'),
    ('2025-03-12_17-21-53', 'gpt-4o-mini', 'human-mouse', 'anatomy'),
    ('2025-03-12_14-43-32', 'gpt-4o-mini', 'omim-ordo', 'bioml-2024')
 ]

suffix = "_reduced"

dfs = []
for run_subdir, model, set_name, dataset in runs:
    metrics_df = read_run_metrics_df(run_subdir, suffix)
    for col, value in zip(["Dataset", "SubSet", "Model"], [dataset, set_name, model]):
        if col not in metrics_df.columns:
            metrics_df[col] = value
    metrics_df["Run"] = run_subdir
    dfs.append(metrics_df)

results_df = pd.concat(dfs, ignore_index=True)
results_df = results_df.sort_values(by=["Dataset", "SubSet", "Model"])

results_df.to_csv(format_combined_metrics_path(suffix), index=False)
results_df

### Rerun evaluation

In [ ]:
runs = [
    ('2025-05-01_11-47-16', 'gemini-2.0-flash', 'snomed-nci', 'largebio'),
    ('2025-05-01_11-37-30', 'gemini-2.0-flash-lite', 'snomed-nci', 'largebio'),
    ('2025-05-01_11-24-31', 'gemini-2.0-flash', 'fma-snomed', 'largebio'),
    ('2025-05-01_11-19-48', 'gemini-2.0-flash-lite', 'fma-snomed', 'largebio'),
    ('2025-05-01_11-15-05', 'gemini-2.0-flash', 'snomed-ncit.pharm', 'bioml-2024'),
    ('2025-05-01_11-12-35', 'gemini-2.0-flash-lite', 'snomed-ncit.pharm', 'bioml-2024'),
    ('2025-05-01_11-07-00', 'gemini-2.0-flash', 'snomed-ncit.neoplas', 'bioml-2024'),
    ('2025-05-01_11-06-39', 'gemini-2.0-flash-lite', 'snomed-ncit.neoplas', 'bioml-2024'),
    ('2025-05-01_10-56-46', 'gemini-2.0-flash', 'snomed-fma.body', 'bioml-2024'),
    ('2025-05-01_10-50-33', 'gemini-2.0-flash-lite', 'snomed-fma.body', 'bioml-2024'),
    ('2025-05-01_10-13-02', 'gemini-2.0-flash-lite', 'fma-snomed', 'largebio'),
    ('2025-04-10_22-11-36', 'gemini-2.0-flash', 'ncit-doid', 'bioml-2024'),
    ('2025-04-10_00-33-19', 'gemini-2.0-flash-lite', 'ncit-doid', 'bioml-2024'),
    ('2025-04-06_22-05-02', 'gemini-2.0-flash-lite', 'omim-ordo', 'bioml-2024'),
    ('2025-04-06_19-27-41', 'gemini-2.0-flash', 'omim-ordo', 'bioml-2024'),
    ('2025-04-06_14-51-00', 'gemini-2.0-flash-lite', 'fma-nci', 'largebio'),
    ('2025-04-06_14-18-46', 'gemini-2.0-flash', 'fma-nci', 'largebio'),
    ('2025-04-04_19-41-28', 'gemini-2.0-flash', 'human-mouse', 'anatomy'),
    ('2025-04-04_19-23-47', 'gemini-2.0-flash-lite', 'human-mouse', 'anatomy'),
    ('2025-04-04_00-43-16', 'gemini-2.0-flash', 'fma-snomed', 'largebio'),
    ('2025-03-28_10-15-04', 'gpt-4o-mini', 'fma-nci', 'largebio'),
    ('2025-03-19_23-32-20', 'gpt-4o-mini', 'omim-ordo', 'bioml-2024'),
    ('2025-03-19_22-17-52', 'gpt-4o-mini', 'ncit-doid', 'bioml-2024'),
    ('2025-03-19_19-48-41', 'gpt-4o-mini', 'human-mouse', 'anatomy'),
    ('2025-03-14_00-40-55', 'gpt-4o-mini', 'ncit-doid', 'bioml-2024'),
    ('2025-03-12_17-21-53', 'gpt-4o-mini', 'human-mouse', 'anatomy'),
    ('2025-03-12_14-43-32', 'gpt-4o-mini', 'omim-ordo', 'bioml-2024')
 ]

In [ ]:
suffix = "_reduced"
for run_subdir, model, set_name, dataset_name in tqdm(runs[:]):
    run_path = RUN_DIR / run_subdir
    print(f"Run path: {run_path}, dataset: {dataset_name}, set: {set_name}, model: {model}")

    for prompt_name in tqdm(PROMPT_FUNCTIONS_MAP, disable=True):
        try:
            prediction_path, stats_path, diagram_path = format_storing_pathes_from_run_path(
                run_path, set_name, model, prompt_name, suffix
            )
            predictions = get_predictions_with_gt(run_path, dataset_name, set_name, model, prompt_name, suffix)
            analyze_results(
                predictions,
                print_results=False,
                plot_confusion_matrix=False,
                subtitle=f"{set_name}: {model} {prompt_name} | ",
                cm_save_path=diagram_path,
                stats_path=stats_path,
            )
        except Exception as e:  # noqa: BLE001
            print(f"Error: {e!s}")

    store_run_metrics_df(PROMPT_FUNCTIONS_MAP, run_path, dataset_name, set_name, model, suffix)